# Frame-window metric comparison
Run all cells to score saved predictions, then compare methods across frame counts. Each point is an equal-weight mean over windows with status `ok`. K is the number of input frames. M counts skipped frames within a window. Start stride controls the separation between window starts.

Fast version: all metrics of one K are scored in a single run of `score_windows_fast.py` (parallel windows, each prediction read once, shared GT KD-trees). It writes the same report folders as `score_windows.py`, so saved results of either are reused.

In [ ]:
from pathlib import Path
import json
import numpy as np
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/score_windows.py").is_file())
DATASET, SPLIT = "eds", "test"
EXPERIMENTS = {2: "windows_k2", 5: "windows_k5", 10: "windows_k10"}
M, START_STRIDE = 0, 50
sys.path.insert(0, str(ROOT / "benchmarks"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "experiments"))
import run_metric as benchmark
# Metrics to score and plot. Comment out what you do not need; every name must be in benchmark.UNITS.
METRICS = [
    "chamfer_full",
    "event_count", "event_count_ratio", "event_rate", "on_percentage", "same_ts",
    "isi_distribution", "isi_median_ms", "isi_cv", "iei_distance", "fano_factor",
    "voxel_mae", "voxel_rmse", "voxel_relative_l1",
    "pmse_2_v2ce", "pmse_4_v2ce",
    # "chamfer", "chamfer_penalty", "chamfer_count",
]
unknown = set(METRICS) - set(benchmark.UNITS)
assert not unknown, f"Unknown metrics: {sorted(unknown)}; choose from {list(benchmark.UNITS)}"
# Window statuses counted as valid in the means, per metric (all others: "ok" only).
# fano_factor is defined on tiles active in both streams; "partial_support" only says some
# tiles were active in one stream alone, so its value is still the metric's value.
VALID_STATUS = {"fano_factor": ("ok", "partial_support")}
BENCHMARK_METHODS = list(benchmark.v2.old.METHODS)
METHODS = []  # Empty discovers available methods for each experiment.
SEQUENCES = []  # Empty evaluates all sequences in the split.
WORKERS, DEVICE = 16, "cpu"  # "cuda": Chamfer on CuPy (same definition, last digits may differ; separate reports)
PROCESSES = None  # Parallel scoring processes; None uses every allocated CPU
PLOT_ONLY = False  # True reads completed reports without starting a scorer.
REPORT_IDS = {}  # Optional: {(2, "chamfer_full"): "12-character report folder"}
REPORTS = {}

## Score windows or open saved reports
Set PLOT_ONLY to True to plot previously saved results, including after restarting the kernel. If several matching configurations exist, select the report folder ID in REPORT_IDS. Empty method and sequence filters allow previously saved subsets; inspect config.json for their coverage.

In [ ]:
REPORTS.clear()
for k, experiment in EXPERIMENTS.items():
    for metric in (METRICS if PLOT_ONLY else []):
        if PLOT_ONLY:
            base = ROOT / "reports" / "experiments" / DATASET / SPLIT / experiment / f"k_{k}"
            if M:
                base /= f"m_{M}"
            candidates = []
            for config_path in sorted((base / metric).glob("*/config.json")):
                folder = config_path.parent
                config = json.loads(config_path.read_text())
                if config["start_stride"] != START_STRIDE or config["m"] != M:
                    continue
                saved_methods = set(config["methods"]) - {"ground_truth"}
                if METHODS and saved_methods != set(METHODS):
                    continue
                if SEQUENCES and set(config["sequences"]) != set(SEQUENCES):
                    continue
                selected_id = REPORT_IDS.get((k, metric))
                if selected_id and folder.name != selected_id:
                    continue
                if all((folder / f).is_file() for f in ("summary.csv", "per_window.csv")):
                    candidates.append(folder)
            if len(candidates) > 1:
                ids = [p.name for p in candidates]
                raise ValueError(f"Select REPORT_IDS[{(k, metric)!r}] from {ids} (see config.json in {base / metric}).")
            if candidates:
                REPORTS[k, metric] = candidates[0]
                print(f"Loaded K={k}, {metric}: {candidates[0].name}")
            else:
                print(f"No saved report for K={k}, {metric}")
            continue
    if PLOT_ONLY:
        continue
    command = [sys.executable, "-u", "experiments/score_windows_fast.py", DATASET,
               "--split", SPLIT, "--experiment-name", experiment, "--k", str(k),
               "--m", str(M), "--start-stride", str(START_STRIDE),
               "--metrics", *METRICS, "--workers", str(WORKERS), "--device", DEVICE]
    if PROCESSES:
        command.extend(["--processes", str(PROCESSES)])
    if METHODS:
        command.extend(["--methods", *METHODS])
    if SEQUENCES:
        command.extend(["--sequences", *SEQUENCES])
    print(f"Computing {len(METRICS)} metrics for K={k}", flush=True)
    last = ""
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, text=True) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            if line.strip():
                last = line.strip()
        status = proc.wait()
    try:
        folders = json.loads(last)
    except ValueError:
        print(f"Unavailable: K={k}. See scorer output above.", flush=True)
        continue
    for metric, folder in folders.items():
        folder = Path(folder)
        if all((folder / f).is_file() for f in ("summary.csv", "per_window.csv")):
            REPORTS[k, metric] = folder
        else:
            print(f"Unavailable: K={k}, {metric}.", flush=True)
    if status:
        print("Scoring errors recorded. Inspect per-window status before interpreting means.")


## Means and coverage
Inspect valid and expected window counts alongside each mean. Different methods can have different valid windows even when their counts match.

In [ ]:
if not REPORTS:
    raise RuntimeError('No reports. Check the experiment names and scorer output above.')
summary = pd.concat([pd.read_csv(p / "summary.csv") for p in REPORTS.values()], ignore_index=True)
windows = pd.concat([pd.read_csv(p / "per_window.csv") for p in REPORTS.values()], ignore_index=True)
# Recompute means for metrics whose VALID_STATUS admits more than "ok" (same equal-weight window mean).
for metric, statuses in VALID_STATUS.items():
    rows = windows[(windows.metric == metric) & windows.status.isin(statuses) & windows.value.notna()]
    stats = rows.groupby(["k", "method"]).value.agg(["mean", "count"])
    target = summary.metric == metric
    keys = list(zip(summary.loc[target, "k"], summary.loc[target, "method"]))
    summary.loc[target, "mean"] = [stats["mean"].get(key, np.nan) for key in keys]
    summary.loc[target, "n_valid"] = [int(stats["count"].get(key, 0)) for key in keys]
display(summary[["k", "metric", "method", "mean", "unit", "n_valid", "n_total", "status_counts"]]
        .sort_values(["metric", "method", "k"]))

## Model means by K
Dark, medium and light blue denote the first three distinct scores. Ties share a color. Values are the saved window means. For descriptive metrics the color ranks distance from the displayed GT mean. For the count ratio it ranks distance from 1. For error metrics smaller means rank first. GT is excluded from ranking. A dash denotes an unavailable scalar; see coverage above. ISI distribution is shown separately as a curve.

In [ ]:
model_order = METHODS or BENCHMARK_METHODS
scalar_metrics = [m for m in METRICS if m != "isi_distribution"]
palette = {1: ("#18548c", "white"), 2: ("#82b7df", "#10263b"), 3: ("#d9ebfa", "#10263b")}

def top_three(column):
    scores = column.drop(index="ground_truth").copy()
    if column.name in benchmark.DESCRIPTIVE:
        scores = (scores - column.at["ground_truth"]).abs()
    elif column.name == "event_count_ratio":
        scores = (scores - 1).abs()
    ranks = scores.rank(method="dense")
    styles = pd.Series("", index=column.index)
    for model, rank in ranks.dropna().items():
        if rank <= 3:
            bg, fg = palette[int(rank)]
            styles[model] = f"background-color: {bg}; color: {fg}"
    return styles

for k in sorted(EXPERIMENTS):
    table = (summary.loc[summary.k == k].pivot(index="method", columns="metric", values="mean")
             .reindex(index=["ground_truth", *model_order], columns=scalar_metrics))
    from html import escape
    from IPython.display import HTML
    styles = table.apply(top_three, axis=0)
    header = "".join(f"<th>{escape(col)}</th>" for col in table.columns)
    body = []
    for method, row in table.iterrows():
        cells = []
        for metric, value in row.items():
            label = "—" if pd.isna(value) else f"{value:.5g}"
            cells.append(f'<td style="{styles.at[method, metric]}">{label}</td>')
        body.append(f"<tr><th>{escape(method)}</th>{''.join(cells)}</tr>")
    display(HTML(f'<div style="overflow-x:auto"><table><caption>K = {k} | Mean over valid windows</caption>'
                 f"<thead><tr><th>Method</th>{header}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"))

## Compare K within each metric
One line per method. Gaps indicate an unavailable mean. A dotted reference at 1 indicates the target for event-count ratio.

In [ ]:
ks = sorted(EXPERIMENTS)
methods = sorted(summary.method.unique())
colors = dict(zip(methods, plt.get_cmap("tab20").colors))
for metric in [m for m in METRICS if m != 'isi_distribution']:
    rows = summary.loc[summary.metric == metric]
    if rows.empty:
        continue
    fig, ax = plt.subplots(figsize=(9, 4))
    for method, group in rows.groupby("method"):
        values = group.set_index("k")["mean"].reindex(ks)
        if values.notna().any():
            if method == "ground_truth":
                ax.plot(ks, values, "k--s", linewidth=2.5, label="GT (recorded)", zorder=5)
            else:
                ax.plot(ks, values, marker="o", color=colors[method], label=method)
    if metric == "event_count_ratio":
        ax.axhline(1, linestyle=":", color="black", label="Target ratio = 1")
    elif metric not in benchmark.DESCRIPTIVE:
        ax.axhline(0, linestyle=":", color="grey", label="Error target = 0")
    ax.set(xticks=ks, xlabel="Input frames per window (K)", ylabel=rows.unit.iloc[0], title=metric)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    else:
        ax.text(.5, .5, "No valid scalar means", ha="center", transform=ax.transAxes)
    ax.grid(axis="y", alpha=.25)
    plt.tight_layout()
    plt.show()

## ISI distribution by K
Each curve is the mean of the per-window empirical cumulative distributions over windows with at least one interval. The grid spans 0 to the longest evaluated window in milliseconds.

In [ ]:
for k in sorted(EXPERIMENTS):
    folder = REPORTS.get((k, "isi_distribution"))
    if folder is None:
        continue
    rows = windows[(windows.k == k) & (windows.metric == "isi_distribution") & (windows.status == "ok")]
    if rows.empty:
        continue
    upper = max(1e-6, rows.duration_s.max() * 1000)
    grid = np.r_[0, np.geomspace(min(1e-6, upper), upper, 160)]
    fig, ax = plt.subplots(figsize=(8, 3.8))
    for method, group in rows.groupby("method"):
        total = np.zeros_like(grid)
        count = 0
        for rel in group.array_path.dropna():
            with np.load(folder / rel) as data:
                intervals = np.sort(data["isi_ms"])
            if intervals.size:
                total += np.searchsorted(intervals, grid, side="right") / intervals.size
                count += 1
        if count:
            gt = method == "ground_truth"
            ax.plot(grid, total/count, "--" if gt else "-", color="black" if gt else colors[method],
                    label="GT (recorded)" if gt else method, zorder=5 if gt else 2)
    ax.set_xscale("symlog", linthresh=0.001)
    ax.set(xlabel="ISI (ms)", ylabel="Mean cumulative fraction", title=f"ISI distribution | K = {k}")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

## Event rate vs time: GT and methods for each K
Interactive plots (zoom, hover, Linear/Log Y) of events per second in the window that starts at `RATE_FIRST`, one figure per K. Windows of every K start at the same frame, so their time axes coincide. V2V is omitted (its voxels hold no event times).

In [ ]:
RATE_SEQUENCE = None   # None: first sequence in the split
RATE_FIRST = 0         # First frame of the window; must be a window start (a multiple of START_STRIDE)
RATE_KS = None         # None: every K in EXPERIMENTS
RATE_METHODS = None    # None: METHODS, or every method with saved predictions
RATE_BIN_MS = 1.0      # Rate bin width in milliseconds
RATE_ON_OFF = False    # Also plot ON and OFF rates separately
RATE_SAVE_HTML = False # Also save each figure under reports/experiments/<dataset>/<split>/event_rates/

from types import SimpleNamespace
from html import escape
from IPython.display import HTML
from score_windows import Window
from benchmarks.event_rate_views_v2 import make_rate_figures

data_root = Path(benchmark.v2.old.setting("EVTWEEN_DATA", benchmark.v2.old.LAB / "evtween/data"))
split_dir = data_root / DATASET / SPLIT
sequence = RATE_SEQUENCE or sorted(p.name for p in split_dir.iterdir() if p.is_dir())[0]
source = split_dir / sequence
files, stamps, _, base, _ = benchmark.v2.sequence_info(source)
if RATE_FIRST % START_STRIDE:
    raise ValueError(f"RATE_FIRST must be a multiple of START_STRIDE={START_STRIDE}")
pred_root = ROOT / "data_test_exp" / DATASET
methods_for_rate = RATE_METHODS or METHODS or [m for m in BENCHMARK_METHODS if (pred_root / m / sequence).is_dir()]
check = SimpleNamespace(dataset=DATASET, split=SPLIT)
raw = {}
for k in (RATE_KS or sorted(EXPERIMENTS)):
    indices = RATE_FIRST + np.arange(k) * (M + 1)
    if indices[-1] >= len(files):
        print(f"K={k}: the window from frame {RATE_FIRST} ends after the sequence ({len(files)} frames).")
        continue
    label = f"frames_{indices[0]}_{indices[-1]}"
    variant = Path(EXPERIMENTS[k]) / f"k_{k}" / (f"m_{M}" if M else "")
    window = Window((source, files, stamps, base), indices, raw)
    edges, meta = window.geometry
    streams, notes = {"Ground truth": window.gt}, []
    for method in methods_for_rate:
        if method == "v2v":
            continue
        folder = pred_root / method / sequence / variant / label
        if not (folder / "events.npz").is_file():
            notes.append(f"{method}: no saved window")
            continue
        try:
            benchmark.verify_metadata(folder / "input.json", meta, check, sequence)
            streams[method] = benchmark.read_events(folder / "events.npz", meta)
        except Exception as exc:
            notes.append(f"{method}: omitted ({exc})")
    title = f"{DATASET}/{SPLIT}/{sequence} · K={k}" + (f", M={M}" if M else "") + f" · {label} · {RATE_BIN_MS:g} ms bins"
    print(title)
    for note in notes:
        print("  ", note)
    on_off, total, _ = make_rate_figures(streams, 0.0, float(edges[-1]), RATE_BIN_MS, title)
    for name, fig in [("total", total)] + ([("on-off", on_off)] if RATE_ON_OFF else []):
        display(HTML(fig.to_html(full_html=False, include_plotlyjs=True, config={"responsive": True, "displaylogo": False})))
        if RATE_SAVE_HTML:
            out = (ROOT / "reports/experiments" / DATASET / SPLIT / "event_rates" /
                   f"{sequence}_k{k}_m{M}_{label}_{RATE_BIN_MS:g}ms_{name}.html")
            out.parent.mkdir(parents=True, exist_ok=True)
            fig.write_html(out, include_plotlyjs=True)
            print("   saved", out)


## Inspect individual windows
Change the filters to inspect individual values and missing or partial scores.

In [ ]:
SHOW_K, SHOW_METRIC = min(EXPERIMENTS), METRICS[0]
SHOW_METHOD, SHOW_SEQUENCE = None, None
selected = windows[(windows.k == SHOW_K) & (windows.metric == SHOW_METRIC)]
if SHOW_METHOD:
    selected = selected[selected.method == SHOW_METHOD]
if SHOW_SEQUENCE:
    selected = selected[selected.sequence == SHOW_SEQUENCE]
display(selected.sort_values(["sequence", "first", "method"]))